# 🔎 Notebook 2: Server-side Discovery + Failure Handling


## 🛠️ Setup

```bash
cd 05-microservices/service-discovery
uv sync
```

Select the `.venv` kernel in VS Code (top-right of the notebook). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


In [ ]:
import time, random

class Registry:
    def __init__(self, ttl=3):
        self.services, self.ttl = {}, ttl
    def register(self, name, addr):
        self.services.setdefault(name, {})[addr] = time.time()
    def heartbeat(self, name, addr):
        self.services[name][addr] = time.time()
    def healthy(self, name):
        now = time.time()
        return [a for a, hb in self.services.get(name, {}).items() if now-hb < self.ttl]

class ServerSideRouter:
    """Client calls us; we consult the registry."""
    def __init__(self, registry):
        self.r = registry
        self.idx = 0
    def route(self, service, path):
        instances = self.r.healthy(service)
        if not instances:
            return {'status': 503, 'error': 'no healthy instances'}
        self.idx = (self.idx + 1) % len(instances)
        target = instances[self.idx]
        return {'status': 200, 'routed_to': target, 'path': path}


## Simulate 3 instances. Crash one. Watch traffic shift.

In [ ]:
reg = Registry(ttl=2)
for addr in ['svc-1', 'svc-2', 'svc-3']:
    reg.register('users', addr)

router = ServerSideRouter(reg)

for _ in range(5):
    print(router.route('users', '/me'))

print('\n--- svc-2 crashes, stops heart-beating ---\n')
# svc-2 stops heart-beating. Wait for every entry to expire, then refresh
# only svc-1 and svc-3 — so svc-2 remains stale and is filtered out.
time.sleep(2.1)
for addr in ['svc-1', 'svc-3']:
    reg.heartbeat('users', addr)

for _ in range(5):
    print(router.route('users', '/me'))


### Client-side vs server-side — when to pick which

| | Client-side | Server-side |
|--|--|--|
| Client complexity | higher (embeds discovery logic) | lower |
| Latency | 1 hop | 2 hops |
| Polyglot fleets | every language needs a client library | just an HTTP hop |
| Examples | Netflix Eureka + Ribbon | AWS ALB, Kubernetes Service, Consul Connect |

### Key ideas
- **Heartbeats + TTL** let the registry forget dead instances automatically.
- **Deregister on shutdown** for instant cleanup; heartbeat TTL is the safety net for crashes.